**Geochemistry Biplot App for Bruker Results.csv Files**

Jupyter Notebook Version
N. Tripcevich 2026, CC BY-SA 4.0  
[More Information Online](https://github.com/arf-berkeley/bruker-xrf-ppm-plot)

For basic use click here, then proceed through this notebook cell-by-cell by pressing Shift-Return on your keyboard. Follow the instructions provided to upload your .csv file and view the data.

The Python in this live notebook can be edited and run again.

In [1]:
%%capture
%pip install plotly ipywidgets
import sys
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.colors import DEFAULT_PLOTLY_COLORS
from IPython.display import display
import ipywidgets as widgets
import io

**Select the Results.csv table from your Bruker analysis**

Browse to a copy of the __Results.csv__ file typically found in Bruker/Data/Results.csv

The code below imports Method Weight % results from the most recent analysis from the end of the Results.csv file.

In [2]:
# Cell 2 - CSV Import - supports both local and hosted environments
from io import StringIO

study_import          = None
study_import_filtered = None

# ── environment detection ─────────────────────────────────────────────────
def is_local():
    try:
        import tkinter as tk
        root = tk.Tk()
        root.destroy()
        return True
    except Exception:
        return False

# ── parser ────────────────────────────────────────────────────────────────
def parse_results_csv(content_str):
    content_str = content_str.replace('\r\n', '\n').replace('\r', '\n')
    lines = [l for l in content_str.split('\n') if l.strip()]

    segment_starts = [
        i for i, line in enumerate(lines)
        if line.split(',')[0].strip().strip('"') == 'File #'
    ]

    if not segment_starts:
        raise ValueError('No "File #" header row found.')

    frames = []
    for idx, start in enumerate(segment_starts, start=1):
        end           = segment_starts[idx] if idx < len(segment_starts) else len(lines)
        segment_lines = lines[start:end]
        if len(segment_lines) < 2:
            continue
        try:
            df_seg = pd.read_csv(
                StringIO('\n'.join(segment_lines)),
                dtype=str,
                skipinitialspace=True
            )
        except Exception as e:
            print(f'  Segment {idx} skipped: {e}')
            continue

        df_seg.dropna(how='all', inplace=True)
        df_seg.dropna(axis=1, how='all', inplace=True)
        if df_seg.empty:
            continue

        df_seg['_batch'] = idx
        frames.append(df_seg)

    if not frames:
        raise ValueError('No data found after parsing all segments.')

    df = pd.concat(frames, ignore_index=True, join='outer')
    df.replace({'< LOD': None, 'None': None, '': None}, inplace=True)

    apps = df['Application'].dropna().unique().tolist() if 'Application' in df.columns else []
    print(f'✓ Loaded {len(df)} rows | {df["_batch"].nunique()} segments | Applications: {apps}')
    return df

# ── filter UI ─────────────────────────────────────────────────────────────
def build_filter_ui():
    global study_import_filtered
    study_import_filtered = study_import.copy()

    def get_unique(col):
        if col not in study_import.columns:
            return []
        return sorted(
            study_import[col]
            .dropna()
            .astype(str)
            .str.strip()
            .replace('', pd.NA)
            .dropna()
            .unique()
            .tolist()
        )

    def batches_for(application):
        df = study_import.copy()
        if application != 'All applications':
            df = df[df['Application'].astype(str).str.strip() == application]
        return ['All'] + [str(b) for b in sorted(df['_batch'].dropna().unique())]

    all_applications = ['All applications'] + get_unique('Application')
    default_app      = all_applications[1] if len(all_applications) > 1 else 'All applications'

    app_dd = widgets.Dropdown(
        options=all_applications,
        value=default_app,
        description='Application:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='360px')
    )
    batch_dd = widgets.Dropdown(
        options=batches_for(default_app),
        value='All',
        description='Batch:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='180px')
    )
    apply_btn   = widgets.Button(
        description='Apply Filter',
        button_style='primary',
        icon='filter',
        layout=widgets.Layout(width='150px')
    )
    summary_out = widgets.Output()
    filter_out  = widgets.Output()

    def refresh_summary(application):
        with summary_out:
            summary_out.clear_output(wait=True)
            df = study_import.copy()
            if application != 'All applications':
                df = df[df['Application'].astype(str).str.strip() == application]
            batches = sorted(df['_batch'].unique().tolist())
            dates   = pd.to_datetime(df['DateTime'], errors='coerce').dropna()
            d_min   = dates.min().strftime('%m-%d-%Y') if not dates.empty else '?'
            d_max   = dates.max().strftime('%m-%d-%Y') if not dates.empty else '?'
            print(f'  "{application}" \u2192 {len(df)} rows')
            print(f'  Batch(es) : {batches}')
            print(f'  Date range: {d_min} \u2013 {d_max}')

    def on_app_change(change):
        # Update batch options without triggering observe side effects
        batch_dd.unobserve_all()
        batch_dd.options = batches_for(change['new'])
        batch_dd.value   = 'All'
        refresh_summary(change['new'])

    def on_apply(btn):
        global study_import_filtered
        filter_out.clear_output(wait=True)
        with filter_out:
            df        = study_import.copy()
            sel_app   = app_dd.value
            sel_batch = batch_dd.value

            if sel_app != 'All applications':
                df = df[df['Application'].astype(str).str.strip() == sel_app]
            if sel_batch != 'All':
                df = df[df['_batch'] == int(sel_batch)]

            df = df.reset_index(drop=True)
            study_import_filtered = df

            if df.empty:
                print(f'No rows for application="{sel_app}" batch="{sel_batch}"')
                print(f'Applications in file : {get_unique("Application")}')
                print(f'Batches in file      : {sorted(study_import["_batch"].unique().tolist())}')
                return

            dates = pd.to_datetime(df['DateTime'], errors='coerce').dropna()
            d_min = dates.min().strftime('%m-%d-%Y %H:%M') if not dates.empty else '?'
            d_max = dates.max().strftime('%m-%d-%Y %H:%M') if not dates.empty else '?'

            try:
                file_nums = df['File #'].dropna().astype(int)
                f_min     = int(file_nums.min())
                f_max     = int(file_nums.max())
            except Exception:
                f_min = df['File #'].dropna().min() if 'File #' in df.columns else '?'
                f_max = df['File #'].dropna().max() if 'File #' in df.columns else '?'

            print(f'\u2713 {len(df)} rows kept')
            print(f'  Application : {sel_app}')
            print(f'  Batch       : {sel_batch}')
            print(f'  File # range: {f_min} \u2013 {f_max}')
            print(f'  Date range  : {d_min} \u2013 {d_max}')
            print('\nNow run Cell 3 \u2192 Cell 4 \u2192 Cell 5')

    app_dd.observe(on_app_change, names='value')
    apply_btn.on_click(on_apply)

    display(widgets.HTML('<b>Filter by Application and Batch</b>'))
    display(widgets.HTML(
        '<span style="color:grey;font-size:12px">'
        'Choose an Application \u2014 Batch will update automatically. '
        'Then click Apply Filter.</span>'
    ))
    display(widgets.VBox([
        widgets.HBox([app_dd, batch_dd, apply_btn]),
        summary_out,
        filter_out
    ]))

    # prime summary once on load without touching batch_dd
    refresh_summary(app_dd.value)

# ── file loading ──────────────────────────────────────────────────────────
if is_local():
    import tkinter as tk
    from tkinter import filedialog

    load_output = widgets.Output()
    display(load_output)

    if study_import is None:
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        study_path = filedialog.askopenfilename(
            title='Select Results.csv',
            filetypes=[('CSV files', '*.csv'), ('All files', '*.*')]
        )
        root.destroy()

        with load_output:
            if study_path:
                with open(study_path, 'r', encoding='utf-8', errors='replace') as f:
                    study_import = parse_results_csv(f.read())
                print(f'  {study_path}\n')
                build_filter_ui()
            else:
                print('No file selected — re-run this cell to try again.')
    else:
        with load_output:
            print(f'Already loaded: {study_import.shape[0]} rows\n')
            build_filter_ui()

else:
    upload_widget = widgets.FileUpload(accept='.csv', multiple=False)
    load_btn      = widgets.Button(
        description='Load Data',
        button_style='success',
        icon='check',
        disabled=True
    )
    status_lbl  = widgets.Label('Upload a Bruker XRF Results.csv file')
    filter_area = widgets.Output()

    def _on_upload_change(change):
        load_btn.disabled = not bool(upload_widget.value)
        if upload_widget.value:
            status_lbl.value = 'File ready \u2014 click Load Data'

    def _on_load(btn):
        global study_import
        try:
            raw          = upload_widget.value[0]['content'].tobytes()
            content      = raw.decode('utf-8', errors='replace')
            study_import = parse_results_csv(content)
            status_lbl.value     = f'\u2713 Loaded {study_import.shape[0]} rows'
            load_btn.disabled    = True
            load_btn.description = 'Loaded'
            with filter_area:
                filter_area.clear_output(wait=True)
                build_filter_ui()
        except Exception as e:
            status_lbl.value = f'Error: {e}'

    upload_widget.observe(_on_upload_change, names='value')
    load_btn.on_click(_on_load)

    display(widgets.VBox([
        widgets.Label('Upload Results.csv:'),
        upload_widget,
        load_btn,
        status_lbl,
        filter_area
    ]))

Output()

***[Click Here to Continue]***

__Clean up Bruker data__

Cleaning data includes removing the following: elemental error columns, Alloy, Match Qual columns, Multiplier, Cal Check, Operator, Field 1&2. This script also replaces Below Detection Limits LOD with 0.

In [3]:
# Cell 3 - Data cleaning
if study_import_filtered is None or len(study_import_filtered) == 0:
    print('Please apply a filter in Cell 2 before continuing.')
else:
    _source = study_import_filtered.copy()
    print(f'Using filtered data : {len(_source)} rows')
    print(f'Application(s)      : {_source["Application"].dropna().unique().tolist()}')
    print(f'Batch(es)           : {sorted(_source["_batch"].unique().tolist())}')

    NON_ELEMENT_COLS = [
        'Alloy 1', 'Match Qual 1', 'Alloy 2', 'Match Qual 2',
        'Alloy 3', 'Match Qual 3', 'Multiplier', 'Cal Check',
        'Operator', 'Field1', 'Field2', 'ID', '_batch', '_method'
    ]

    drop_cols = [c for c in _source.columns if 'Err' in c or c in NON_ELEMENT_COLS]
    keep_cols = [c for c in _source.columns if c not in drop_cols]
    study     = _source[keep_cols].copy()

    study = study.replace('< LOD', 0)

    string_cols  = [c for c in ['Name', 'Application', 'Method'] if c in study.columns]
    numeric_cols = [c for c in study.columns if c not in string_cols + ['DateTime']]
    study[string_cols]  = study[string_cols].astype('string')
    study[numeric_cols] = study[numeric_cols].apply(pd.to_numeric, errors='coerce')
    study['DateTime']   = pd.to_datetime(study['DateTime'], errors='coerce')

    element_cols = [c for c in numeric_cols if c not in ['File #', 'ElapsedTime']]
    study[element_cols] = (study[element_cols] * 10000).round(1)

    study.dropna(axis=1, how='all', inplace=True)
    study.dropna(how='all', inplace=True)

    print(f'Rows after cleaning : {len(study)}')
    print(f'Columns             : {study.columns.tolist()}')

Using filtered data : 91 rows
Application(s)      : ['Obsidian 3mm']
Batch(es)           : [11]
Rows after cleaning : 91
Columns             : ['File #', 'DateTime', 'Name', 'Application', 'Method', 'ElapsedTime', 'Mn', 'Zr', 'Rb', 'Sr', 'Y', 'Nb', 'Ba', 'Th']


**Display Data Table**

Run the next cell to view the data table before viewing a biplot.

In [4]:
# Cell 4 - Data Table
if study is None:
    print('Please complete data cleaning before continuing')
else:
    # --- Formatted Table ---
    display(widgets.HTML('<b>Results Table</b>'))
    display_df = study.copy()
    display_df['DateTime'] = display_df['DateTime'].dt.strftime('%m/%d/%Y %H:%M')
    display_df = display_df.rename(columns={'ElapsedTime': 'Elapsed'})
    
    # Identify column types for formatting
    non_element = ['File #', 'DateTime', 'Name', 'Application', 'Method', 'Elapsed']
    text_cols = [c for c in ['DateTime', 'Name', 'Application', 'Method'] if c in display_df.columns]
    
    # Only apply float format to numeric element columns
    element_cols = [c for c in display_df.columns 
                   if c not in non_element 
                   and pd.api.types.is_numeric_dtype(display_df[c])]

    display(display_df.style
        .format({col: '{:.1f}' for col in element_cols})
        .set_properties(**{
            'text-align': 'right',
            'font-size': '12px'
        })
        .set_properties(subset=text_cols, **{
            'text-align': 'left'
        })
        .set_table_styles([{
            'selector': 'th',
            'props': [('text-align', 'center'), ('font-weight', 'bold')]
        }])
        .hide(axis='index')
    )


HTML(value='<b>Results Table</b>')

File #,DateTime,Name,Application,Method,Elapsed,Mn,Zr,Rb,Sr,Y,Nb,Ba,Th
646,05/15/2026 09:29,stone,Obsidian 3mm,Obsidian 3mm,30.000000,823.0,237.0,162.0,133.0,29.0,23.0,1853.0,23.0
647,05/15/2026 10:57,stone,Obsidian 3mm,Obsidian 3mm,30.000000,1066.0,84.0,263.0,54.0,26.0,28.0,779.0,32.0
648,05/15/2026 10:59,stone,Obsidian 3mm,Obsidian 3mm,30.000000,997.0,74.0,244.0,48.0,23.0,31.0,1032.0,39.0
649,05/15/2026 10:59,stone,Obsidian 3mm,Obsidian 3mm,30.000000,1004.0,74.0,234.0,51.0,28.0,31.0,1021.0,37.0
650,05/15/2026 11:00,stone,Obsidian 3mm,Obsidian 3mm,30.000000,726.0,93.0,123.0,113.0,20.0,24.0,1133.0,24.0
651,05/15/2026 11:02,stone,Obsidian 3mm,Obsidian 3mm,30.000000,1163.0,90.0,288.0,57.0,24.0,34.0,696.0,41.0
652,05/15/2026 11:03,stone,Obsidian 3mm,Obsidian 3mm,30.000000,907.0,72.0,227.0,46.0,24.0,28.0,899.0,35.0
653,05/15/2026 11:04,stone,Obsidian 3mm,Obsidian 3mm,30.000000,1104.0,87.0,270.0,51.0,24.0,37.0,1024.0,36.0
654,05/15/2026 11:04,stone,Obsidian 3mm,Obsidian 3mm,30.000000,1016.0,74.0,254.0,48.0,26.0,32.0,801.0,31.0
655,05/15/2026 11:06,stone,Obsidian 3mm,Obsidian 3mm,30.000000,983.0,79.0,267.0,57.0,18.0,35.0,696.0,34.0


In [ ]:
# Cell 5 - Biplot
if study is None:
    print('Please complete data cleaning before continuing')
else:
    NON_ELEMENT_COLS = ['File #', 'DateTime', 'Name', 'Application',
                        'Method', 'ElapsedTime', 'Elapsed', '_batch', '_method']
    elements_present = [
        c for c in study.columns
        if c not in NON_ELEMENT_COLS
        and pd.api.types.is_numeric_dtype(study[c])
    ]

    if len(elements_present) < 2:
        print(f'Not enough numeric element columns to plot. Found: {elements_present}')
    else:
        x_dropdown = widgets.Dropdown(
            options=elements_present,
            value='Sr' if 'Sr' in elements_present else elements_present[0],
            description='X Axis:',
            style={'description_width': 'initial'}
        )
        y_dropdown = widgets.Dropdown(
            options=elements_present,
            value='Rb' if 'Rb' in elements_present else elements_present[1],
            description='Y Axis:',
            style={'description_width': 'initial'}
        )

        plot_output = widgets.Output()

        def update_plot(change):
            with plot_output:
                plot_output.clear_output(wait=True)
                x = x_dropdown.value
                y = y_dropdown.value

                # Make a plain-Python-string copy of study for plotly
                # This avoids the pandas NA ambiguous boolean error
                plot_df = study.copy()

                # Convert all string/object/StringDtype columns to plain str
                for col in plot_df.columns:
                    if hasattr(plot_df[col], 'dtype') and (
                        plot_df[col].dtype == 'string' or
                        plot_df[col].dtype == object or
                        str(plot_df[col].dtype) == 'StringDtype'
                    ):
                        plot_df[col] = plot_df[col].astype(object).where(
                            plot_df[col].notna(), other='(no name)'
                        ).astype(str)

                # Fill any remaining NA in Name
                if 'Name' in plot_df.columns:
                    plot_df['Name'] = plot_df['Name'].replace(
                        {'nan': '(no name)', 'None': '(no name)', '<NA>': '(no name)'}
                    ).fillna('(no name)')

                color_col  = 'Name' if 'Name' in plot_df.columns else None
                name_order = (
                    sorted(plot_df['Name'].unique().tolist())
                    if color_col else []
                )

                # Build hover list from columns that exist and aren't x or y
                hover_data = [
                    c for c in ['File #', 'DateTime']
                    if c in plot_df.columns and c != x and c != y
                ]

                try:
                    fig = px.scatter(
                        plot_df,
                        x=x,
                        y=y,
                        color=color_col,
                        category_orders={'Name': name_order},
                        hover_data=hover_data,
                        title=f'{y} vs {x} Biplot',
                        labels={
                            x: f'{x} (PPM)',
                            y: f'{y} (PPM)',
                            'Name': 'Sample Name'
                        }
                    )
                    fig.update_traces(marker=dict(size=8, opacity=0.85))
                    fig.update_layout(
                        height=600,
                        hovermode='closest',
                        legend=dict(
                            title=dict(text='Sample Name', font=dict(size=13)),
                            itemsizing='constant',
                            bordercolor='lightgrey',
                            borderwidth=1,
                            bgcolor='rgba(255,255,255,0.85)',
                            x=1.02,
                            xanchor='left',
                            y=1,
                            yanchor='top'
                        ),
                        margin=dict(r=180)
                    )
                    fig.show()
                except Exception as e:
                    print(f'Plot error: {e}')
                    print(f'  x={x}, y={y}')
                    print(f'  Name dtype: {plot_df["Name"].dtype if "Name" in plot_df.columns else "missing"}')
                    print(f'  Name sample: {plot_df["Name"].unique()[:5] if "Name" in plot_df.columns else "n/a"}')

        x_dropdown.observe(update_plot, names='value')
        y_dropdown.observe(update_plot, names='value')

        display(widgets.VBox([
            widgets.HBox([x_dropdown, y_dropdown]),
            plot_output
        ]))

        update_plot(None)

In [6]:
# Cell 6 - Ternary Plot
if study is None:
    print('Please complete data cleaning before continuing')
else:
    NON_ELEMENT_COLS = ['File #', 'DateTime', 'Name', 'Application',
                        'Method', 'ElapsedTime', 'Elapsed', '_batch', '_method']
    elements_present = [
        c for c in study.columns
        if c not in NON_ELEMENT_COLS
        and pd.api.types.is_numeric_dtype(study[c])
    ]

    if len(elements_present) < 3:
        print(f'Not enough numeric element columns. Found: {elements_present}')
    else:
        # Default to Rb, Sr, Zr if available
        default_a = 'Rb' if 'Rb' in elements_present else elements_present[0]
        default_b = 'Sr' if 'Sr' in elements_present else elements_present[1]
        default_c = 'Zr' if 'Zr' in elements_present else elements_present[2]

        a_dropdown = widgets.Dropdown(
            options=elements_present,
            value=default_a,
            description='A (top):',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='200px')
        )
        b_dropdown = widgets.Dropdown(
            options=elements_present,
            value=default_b,
            description='B (bottom left):',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='220px')
        )
        c_dropdown = widgets.Dropdown(
            options=elements_present,
            value=default_c,
            description='C (bottom right):',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='220px')
        )

        plot_output = widgets.Output()

        def update_plot(change):
            with plot_output:
                plot_output.clear_output(wait=True)
                a = a_dropdown.value
                b = b_dropdown.value
                c = c_dropdown.value

                if len({a, b, c}) < 3:
                    print('Please select three different elements.')
                    return

                # Clean copy with plain Python strings for plotly
                plot_df = study.copy()
                for col in plot_df.columns:
                    if str(plot_df[col].dtype) in ('string', 'StringDtype') or \
                       plot_df[col].dtype == object:
                        plot_df[col] = (
                            plot_df[col]
                            .astype(object)
                            .fillna('(no name)')
                            .astype(str)
                        )

                if 'Name' in plot_df.columns:
                    plot_df['Name'] = plot_df['Name'].replace(
                        {'nan': '(no name)', 'None': '(no name)', '<NA>': '(no name)'}
                    )

                # Drop rows where any of the three elements are NaN
                plot_df = plot_df.dropna(subset=[a, b, c])

                if plot_df.empty:
                    print(f'No rows with valid data for {a}, {b}, {c}.')
                    return

                color_col  = 'Name' if 'Name' in plot_df.columns else None
                name_order = (
                    sorted(plot_df['Name'].unique().tolist())
                    if color_col else []
                )

                try:
                    fig = px.scatter_ternary(
                        plot_df,
                        a=a,
                        b=b,
                        c=c,
                        color=color_col,
                        category_orders={'Name': name_order},
                        hover_data=[
                            col for col in ['File #', 'DateTime']
                            if col in plot_df.columns
                        ],
                        title=f'Ternary Plot: {a} / {b} / {c}',
                        labels={
                            'Name': 'Sample Name',
                            a: f'{a} (PPM)',
                            b: f'{b} (PPM)',
                            c: f'{c} (PPM)'
                        }
                    )
                    fig.update_traces(marker=dict(size=8, opacity=0.85))
                    fig.update_layout(
                        height=650,
                        legend=dict(
                            title=dict(text='Sample Name', font=dict(size=13)),
                            itemsizing='constant',
                            bordercolor='lightgrey',
                            borderwidth=1,
                            bgcolor='rgba(255,255,255,0.85)',
                            x=1.02,
                            xanchor='left',
                            y=1,
                            yanchor='top'
                        ),
                        margin=dict(r=180)
                    )
                    fig.show()
                except Exception as e:
                    print(f'Plot error: {e}')
                    print(f'  a={a}, b={b}, c={c}')
                    print(f'  Name dtype: {plot_df["Name"].dtype if "Name" in plot_df.columns else "missing"}')

        a_dropdown.observe(update_plot, names='value')
        b_dropdown.observe(update_plot, names='value')
        c_dropdown.observe(update_plot, names='value')

        display(widgets.VBox([
            widgets.HBox([a_dropdown, b_dropdown, c_dropdown]),
            plot_output
        ]))

        update_plot(None)

In [7]:
# Cell 7 - Export dataset as CSV
from IPython.display import display, HTML
if study is None:
    print('Please complete data cleaning before continuing')
else:
    export_output = widgets.Output()

    export_btn = widgets.Button(
        description='Export CSV',
        button_style='success',
        icon='download'
    )

    def on_export(btn):
        with export_output:
            export_output.clear_output(wait=True)
            if is_local():
                import tkinter as tk
                from tkinter import filedialog

                root = tk.Tk()
                root.withdraw()
                root.attributes('-topmost', True)
                save_path = filedialog.asksaveasfilename(
                    title='Save CSV',
                    defaultextension='.csv',
                    filetypes=[('CSV files', '*.csv'), ('All files', '*.*')],
                    initialfile=f'Bruker_Results_export_{study["DateTime"].max().strftime("%Y%m%d")}.csv'
                )
                root.destroy()

                if save_path:
                    study.to_csv(save_path, index=False)
                    print(f'✓ Exported {len(study)} rows to {save_path}')
                else:
                    print('Export cancelled')
            else:
                import base64
                from IPython.display import HTML
                csv_str = study.to_csv(index=False)
                b64 = base64.b64encode(csv_str.encode()).decode()
                filename = f'Bruker_Results_export_{study["DateTime"].max().strftime("%Y%m%d")}.csv'
                html = f'<a download="{filename}" href="data:text/csv;base64,{b64}">Click here to download {filename}</a>'
                display(HTML(html))

    export_btn.on_click(on_export)

    display(widgets.HTML('<b>Would you like to export the cleaned up CSV of these values?</b>'))
    display(widgets.VBox([export_btn, export_output]))

HTML(value='<b>Would you like to export the cleaned up CSV of these values?</b>')